# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to load, explore, and process the publicly available FAIR² dataset defined by a Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata  # Metadata is an object, not dict/list
print(f"{meta.name}: {meta.description}")


## 2. Data Overview

Review available record sets, fields, and their `@id` references.

In [ ]:
# Croissant datasets define record sets, each with a unique `@id`.
print("Available record sets and fields:")

record_sets = list(dataset.record_sets)  # list of mlcroissant.RecordSet

for rs in record_sets:
    print(f'RecordSet name: {rs.name!r}\n  @id: {rs.id}\n  Fields:')
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print()

## 3. Data Extraction

Load data from each record set using its `@id`. Data is returned as a generator of records, which can be loaded into a pandas DataFrame for further analysis.

We will extract all record sets available in this dataset, referencing all by their `@id`.

In [ ]:
# Get all available record sets by @id
record_set_ids = [rs.id for rs in dataset.record_sets]
print("RecordSet @ids found:", record_set_ids)

# Extract all record sets into DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}:", df.columns.tolist())
    print(f"Sample data from {record_set_id}:")
    display(df.head())

# Select a record set for further EDA below.
# If you know the main record set @id you want, set it here, else just pick the first one.
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f\n"Using main record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
else:
    print('No record sets were found in the dataset.')


## 4. Exploratory Data Analysis (EDA)

Apply standard data processing steps such as filtering, normalization, and aggregation using only field and column `@id`s.

In [ ]:
from IPython.display import display

# Use the main record set chosen previously
if main_record_set_id and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]

    # Find numeric fields based on DataFrame dtypes
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use the first numeric field
        print(f"Numeric field selected (by @id): {numeric_field_id}")

        # Filter records where field > threshold
        threshold = df[numeric_field_id].mean()  # Example criterion: greater than mean
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        norm_col = numeric_field_id + '_normalized'
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a non-numeric field if present
        group_field_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]  # Take the first categorical field
            print(f"Grouping by field (by @id): {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found in the selected record set.")
else:
    print('No data available for EDA.')

## 5. Visualization

Visualize data distributions and relationships using the available fields (using only field `@id`s as references).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]
    # Numeric field for plotting
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        # Distribution plot
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id], kde=True)
        plt.title(f"Distribution of '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.show()

        # If there's a categorical/group field, plot boxplot
        group_field_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            plt.figure(figsize=(8,4))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f"{numeric_field_id} by '{group_field_id}'")
            plt.show()
    else:
        print('No numeric fields available for visualization.')
else:
    print('No data available for visualization.')

## 6. Conclusion

This notebook demonstrated how to explore a Croissant-standard FAIR² dataset with multiple record sets and fields using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

- Dataset metadata, tables (`recordSet` and field `@id`s) were explored dynamically.
- Data from each record set was loaded using their `@id`s; further analysis used only these stable identifiers.
- A basic exploratory analysis and example normalization/grouping and visualization were shown.

For deeper domain-specific analysis, review the dataset's field documentation or variables dictionary as defined in the Croissant schema and further visualize or model the data of interest.